# Agent Traces to SFT

Distill a `gpt-4.1-mini` hosted agent into `gpt-4.1-nano` with the original demo's scale, training recipe, evaluator, and gain calculation, without restoring its row-level leakage.

**Primary compatibility contract:** acquire at least 100 distinct conversations, select one canonical repaired snapshot per conversation, split exactly 80/10/10 by conversation, train for 3 epochs with learning-rate multiplier 1.0 and batch size 1, and evaluate the first assistant tool-call action with the original score and pass threshold of 8. Full-trajectory, argument, and extra-call metrics are secondary diagnostics.

**Critical lineage rule:** transformation, privacy validation, splitting, evaluation, and training use only the fresh trace export acquired for the current run. Runtime configuration and evidence are ignored by Git.


## 1. Setup, roles, privacy, provenance, and the runnable path

The hosted `gpt-4.1-mini` agent is the teacher and `gpt-4.1-nano-2025-04-14` is the student. Application Insights is the authorized acquisition source. Remove secrets, credentials, personal data, customer identifiers, and unnecessary free text before training.

The complete bounded live runner is `../live_validate.py`. It generates a fresh conversation buffer, waits for ingestion, acquires traces, selects exactly 100 canonical independent conversations, proves zero split overlap, uploads the exact 80/10 files, submits or exactly matches an existing job, monitors it to terminal state, deploys it, and evaluates teacher, base, and fine-tuned models. Run it from the demo directory:

```powershell
python live_validate.py --fresh-conversations 120 --target-conversations 100
```


In [ ]:
import hashlib
import json
import os
import random
import sys
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()
ROOT = Path.cwd()
DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))
from live_validate import (
    evaluate_deployment,
    prepare_canonical_splits,
    tool_call_score,
)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
AGENT_NAME = os.environ["FOUNDRY_AGENT_NAME"]
AGENT_VERSION = os.environ["FOUNDRY_AGENT_VERSION"]
TEACHER_DEPLOYMENT = os.environ.get("FOUNDRY_TEACHER_DEPLOYMENT", "gpt-4.1-mini")
STUDENT_DEPLOYMENT = os.environ.get("FOUNDRY_STUDENT_DEPLOYMENT", "gpt-4.1-nano")
STUDENT_MODEL = os.environ.get("FOUNDRY_STUDENT_MODEL", "gpt-4.1-nano-2025-04-14")

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential, allow_preview=True)
model_client = project.get_openai_client()
agent = project.agents.get_version(agent_name=AGENT_NAME, agent_version=AGENT_VERSION)
print("Validated:", agent.name, agent.version, TEACHER_DEPLOYMENT, STUDENT_MODEL)


## 2. Acquire enough fresh traces for 100 independent conversations

The original notebook requested 100 transformed trace rows and described approximately 100 conversations, but overlapping snapshots made those units ambiguous. Compatibility here is defined conservatively as **100 independent conversation units**, one canonical snapshot each. The live runner generates 120 unique conversations as an ingestion buffer and requests up to 500 trace rows, then stops unless at least 100 fresh conversation prompts are present.


In [ ]:
# The live runner performs acquisition and records job IDs, output file IDs, timestamps, and hashes.
# This notebook intentionally does not reuse a committed trace snapshot.
print("Run: python live_validate.py --fresh-conversations 120 --target-conversations 100")


## 3. Transform the current trace export

The current export can contain overlapping snapshots, fragments without tool-learning signal,
literal `"null"` assistant content, adjacent tool-call turns, and missing system/tools metadata.
The following idempotent transform repairs those shapes while preserving message order.

In [ ]:
def message_key(message):
    tool_calls = tuple(
        (
            call.get("id"),
            (call.get("function") or {}).get("name"),
            (call.get("function") or {}).get("arguments"),
        )
        for call in message.get("tool_calls", [])
    )
    return message.get("role"), message.get("content"), message.get("tool_call_id"), tool_calls

def deduplicate_messages(messages):
    seen, result = set(), []
    for message in messages:
        key = message_key(message)
        if key not in seen:
            seen.add(key)
            result.append(dict(message))
    return result

def stable_call_id(row_digest, message_index, call_index):
    seed = f"{row_digest}:{message_index}:{call_index}".encode("utf-8")
    return "call_" + hashlib.sha256(seed).hexdigest()[:24]

def repair_tool_call_ids(messages, row_digest):
    repaired = json.loads(json.dumps(messages))
    pending = []
    for message_index, message in enumerate(repaired):
        if message.get("role") == "assistant" and message.get("tool_calls"):
            for call_index, call in enumerate(message["tool_calls"]):
                if not call.get("id"):
                    call["id"] = stable_call_id(row_digest, message_index, call_index)
                pending.append(call["id"])
        elif message.get("role") == "tool":
            tool_call_id = message.get("tool_call_id")
            if tool_call_id in pending:
                pending.remove(tool_call_id)
            elif pending:
                message["tool_call_id"] = pending.pop(0)
            else:
                raise ValueError("Tool response has no preceding unanswered tool call.")
    return repaired

def normalize_null_content(messages):
    for message in messages:
        if message.get("role") == "assistant" and message.get("tool_calls"):
            if message.get("content") in {None, "null", ""}:
                message.pop("content", None)
    return messages

def merge_adjacent_tool_calls(messages):
    merged = []
    for message in messages:
        if (
            merged
            and message.get("role") == "assistant"
            and message.get("tool_calls")
            and merged[-1].get("role") == "assistant"
            and merged[-1].get("tool_calls")
        ):
            merged[-1]["tool_calls"].extend(message["tool_calls"])
        else:
            merged.append(message)
    return merged

def contains_tool_signal(messages):
    return any(message.get("role") == "assistant" and message.get("tool_calls") for message in messages)

def transform_trace_row(row):
    row_digest = hashlib.sha256(json.dumps(row, sort_keys=True).encode("utf-8")).hexdigest()
    messages = repair_tool_call_ids(row.get("messages", []), row_digest)
    messages = merge_adjacent_tool_calls(normalize_null_content(deduplicate_messages(messages)))
    if not contains_tool_signal(messages):
        return None
    if not messages or messages[0].get("role") != "system":
        messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
    return {"messages": messages, "tools": TOOLS}

## 4. Privacy review, canonical conversation construction, and exact split

After deterministic export repair and redaction, group rows by the exact initial user prompt. For each fresh conversation select the longest repaired snapshot, with canonical JSON as the deterministic tie-break. Sort, seed-shuffle with 42, and split exactly 80 train / 10 validation / 10 test conversations. Prompt-hash intersections across all split pairs must be empty.


In [ ]:
# `prepare_canonical_splits` is the implementation used by the live run.
# It writes exact JSONL bytes and a lineage manifest containing counts, SHA-256 hashes,
# privacy results, canonical-selection policy, and zero-overlap proof.
latest = json.loads((OUTPUTS / "latest.json").read_text(encoding="utf-8"))
run_dir = Path(latest["run_dir"])
lineage = json.loads((run_dir / "lineage.json").read_text(encoding="utf-8"))
assert lineage["selected_canonical_conversations"] == 100
assert lineage["split_row_counts"] == {"train": 80, "validation": 10, "test": 10}
assert not any(lineage["conversation_overlap"].values())
lineage


## 5. Original-compatible primary evaluator and strict secondary diagnostics

Primary evaluation reproduces the original context: system prompt plus the first user prompt, temperature 0, and only the first assistant action. `tool_call_score` returns 10 for exact tool names and arguments, 8 for the same tool-name set with argument differences, 2-8 for partial name overlap, and 1 for no overlap or a missing call. A row passes at score >= 8.

Secondary diagnostics execute the full trajectory and report full structural score, exact name sequence, exact trajectory, aligned argument accuracy, and extra calls. These do not replace the primary compatibility result.


In [ ]:
assert tool_call_score(
    [{"name": "get_order_details", "arguments": {"order_id": "ZA-1"}}],
    [{"name": "get_order_details", "arguments": {"order_id": "ZA-1"}}],
) == 10
assert tool_call_score(
    [{"name": "get_order_details", "arguments": {"order_id": "ZA-1"}}],
    [{"name": "get_order_details", "arguments": {"order_id": "ZA-2"}}],
) == 8
print("Original-compatible evaluator loaded; pass threshold = 8")


## 6. Fine-tune, deploy, and compare

Upload the exact hashed 80-row training file and 10-row validation file. Submit supervised SFT on `gpt-4.1-nano-2025-04-14` with 3 epochs, learning-rate multiplier 1.0, batch size 1, and `GlobalStandard`. Reuse is allowed only when uploaded file IDs, model, method, and hyperparameters match exactly. Monitor to a terminal state, deploy the resulting model, and evaluate teacher, base, and fine-tuned deployments on the same 10 held-out conversations.


In [ ]:
summary = json.loads(
    (run_dir / "live_validation_summary.json").read_text(encoding="utf-8")
)
training = summary["training"]
assert training["status"] == "succeeded"
assert training["hyperparameters"] == {
    "n_epochs": 3,
    "learning_rate_multiplier": 1.0,
    "batch_size": 1,
}
evaluation = summary["evaluation"]
evaluation["outcome"]


## 7. Gain calculation and interpretation

The primary reported gain uses the historical formula:

`(fine_tuned_average_score - base_average_score) / base_average_score * 100`

Pass-rate gain is reported separately in percentage points. The historical source run reported 7.38 to 8.60 (+16.5%) and 60% to 100% pass rate. This run uses the same primary methodology at the same 100/80/10/10 scale, while its conversation-level construction prevents overlapping snapshots from leaking between train, validation, and test.

> Actual results may vary by model version, data, configuration, region availability, and service conditions. Preserve every job ID, file ID, count, hash, and metric in ignored runtime evidence.
